# Лабораторная работа №1, часть 2

## Самостоятельное решение задачи линейного программирования двумя способами

Цель этой части: самостоятельно решить **новую** задачу ЛП
- геометрическим методом (вручную + визуализация в Python),
- программным методом через `scipy.optimize.linprog`,
- затем сверить результат через мягкую автопроверку.


## 1. Исходная задача (новый вариант)

Найти максимум целевой функции:

$$
\max z = 7x_1 + 5x_2
$$

при ограничениях:

$$
egin{cases}
2x_1 + x_2 \le 12, \
x_1 + 2x_2 \le 12, \
x_1 \ge 0,\; x_2 \ge 0.
\end{cases}
$$

Где:
- $x_1$ — объём выпуска продукта A,
- $x_2$ — объём выпуска продукта B,
- $z$ — прибыль.


## 2. Задание 1: геометрическое решение (самостоятельно)

Выполните последовательно:

1. Перепишите ограничения в виде граничных прямых (замените $\le$ на $=$).
2. Найдите точки пересечения прямых с осями.
3. Постройте допустимую область (первая четверть + полуплоскости).
4. Найдите все угловые точки допустимой области.
5. Вычислите $z = 7x_1 + 5x_2$ в каждой вершине.
6. Определите оптимальный план и максимум прибыли.

Подсказка: итог ручного решения внесите в переменные `x1_manual`, `x2_manual`, `z_manual` в кодовой ячейке ниже.


In [ ]:
# Впишите сюда результаты СВОЕГО геометрического решения.
# Если ещё не решили вручную, оставьте None и вернитесь позже.

x1_manual = None  # например: 0.0
x2_manual = None  # например: 0.0
z_manual = None   # например: 0.0

print("Ручное решение (черновик):")
print("x1_manual =", x1_manual)
print("x2_manual =", x2_manual)
print("z_manual  =", z_manual)


## 3. Визуализация геометрического решения в Python

Постройте график ограничений и допустимой области. После построения убедитесь, что ваша точка оптимума
из ручного решения действительно лежит в вершине допустимой области и даёт наибольший уровень цели.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Диапазон для построения по оси x1
x1_grid = np.linspace(0, 8, 400)

# Граничные прямые ограничений:
# 2*x1 + x2 = 12  -> x2 = 12 - 2*x1
# x1 + 2*x2 = 12  -> x2 = (12 - x1)/2
line_1 = 12 - 2 * x1_grid
line_2 = (12 - x1_grid) / 2

# Вершины допустимой области (проверьте своим ручным решением)
vertices = np.array([
    [0, 0],
    [6, 0],
    [4, 4],
    [0, 6],
], dtype=float)

fig, ax = plt.subplots(figsize=(8, 6))

# Прямые ограничений
ax.plot(x1_grid, line_1, label=r'$2x_1 + x_2 = 12$', linewidth=2)
ax.plot(x1_grid, line_2, label=r'$x_1 + 2x_2 = 12$', linewidth=2)

# Допустимая область
ax.fill(vertices[:, 0], vertices[:, 1], alpha=0.25, color='tab:green', label='Допустимая область')

# Отметим вершины
ax.scatter(vertices[:, 0], vertices[:, 1], color='black', zorder=3)
for vx, vy in vertices:
    ax.annotate(f'({vx:.0f}, {vy:.0f})', (vx, vy), textcoords='offset points', xytext=(6, 6), fontsize=10)

# Если ручной ответ заполнен — покажем точку ручного оптимума
if x1_manual is not None and x2_manual is not None:
    ax.scatter([x1_manual], [x2_manual], color='crimson', s=90, zorder=4, label='Ваш ручной оптимум')

ax.set_xlim(0, 8)
ax.set_ylim(0, 8)
ax.set_xlabel(r'$x_1$')
ax.set_ylabel(r'$x_2$')
ax.set_title('Задание 1: геометрическая интерпретация')
ax.grid(alpha=0.3)
ax.legend(loc='upper right')
plt.show()


## 4. Задание 2: программное решение через `linprog`

Соберите матричную запись и решите ту же задачу в Python.

Напоминание:
- `linprog` минимизирует, поэтому для максимизации $7x_1 + 5x_2$ используем минимизацию $-7x_1 - 5x_2$.
- После решения восстановите максимум: `z_py = -res.fun`.


In [ ]:
import numpy as np
from scipy.optimize import linprog

# max z = 7*x1 + 5*x2  <=>  min f = -7*x1 - 5*x2
c = np.array([-7.0, -5.0])

# Ограничения A_ub @ x <= b_ub
A_ub = np.array([
    [2.0, 1.0],
    [1.0, 2.0],
])
b_ub = np.array([12.0, 12.0])

# x1 >= 0, x2 >= 0
bounds = [(0, None), (0, None)]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")

print("res.success:", res.success)
print("res.status:", res.status)
print("res.message:", res.message)

x1_py = None
x2_py = None
z_py = None

if res.success:
    x1_py, x2_py = res.x
    z_py = -res.fun
    print(f"x1_py = {x1_py:.6g}")
    print(f"x2_py = {x2_py:.6g}")
    print(f"z_py  = {z_py:.6g}")
else:
    print("Оптимум не найден.")


## 5. Мягкая автопроверка (без раскрытия эталона)

Ячейка проверяет:
- корректность вашего ручного ответа (`x1_manual`, `x2_manual`, `z_manual`),
- корректность Python-ответа (`x1_py`, `x2_py`, `z_py`).

Эталонные значения в вывод не печатаются: будет только статус и указание, где несоответствие.


In [ ]:
import numpy as np

# Внутренний эталон (не выводим его в печать)
_REF = {
    "x1": 4.0,
    "x2": 4.0,
    "z": 48.0,
}


def _to_float_or_nan(v):
    if v is None:
        return np.nan
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def _check_block(label, x1, x2, z):
    x1v = _to_float_or_nan(x1)
    x2v = _to_float_or_nan(x2)
    zv = _to_float_or_nan(z)

    if np.isnan([x1v, x2v, zv]).any():
        print(f"[{label}] FAIL: заполните все три значения (x1, x2, z).")
        return False

    issues = []
    if not np.isclose(x1v, _REF["x1"], atol=1e-6):
        issues.append("x1")
    if not np.isclose(x2v, _REF["x2"], atol=1e-6):
        issues.append("x2")
    if not np.isclose(zv, _REF["z"], atol=1e-6):
        issues.append("z")

    # Дополнительная внутренняя проверка согласованности z = 7*x1 + 5*x2
    if np.isfinite(x1v) and np.isfinite(x2v) and np.isfinite(zv):
        if not np.isclose(zv, 7 * x1v + 5 * x2v, atol=1e-6):
            issues.append("формула z=7*x1+5*x2")

    if not issues:
        print(f"[{label}] PASS")
        return True

    uniq = []
    for name in issues:
        if name not in uniq:
            uniq.append(name)
    print(f"[{label}] FAIL: проверьте -> {', '.join(uniq)}")
    return False

print("=== Автопроверка ===")
ok_manual = _check_block("Геометрическое/ручное", x1_manual, x2_manual, z_manual)
ok_python = _check_block("Python (linprog)", x1_py, x2_py, z_py)

if ok_manual and ok_python:
    print("Итог: оба способа дают корректный результат.")
else:
    print("Итог: есть несоответствия, исправьте и запустите проверку снова.")


## 6. Что написать в выводе

Кратко зафиксируйте в отчёте:

1. Как получена допустимая область и какие у неё угловые точки.
2. Какой оптимум получен геометрически (`x1_manual`, `x2_manual`, `z_manual`).
3. Какой оптимум получен программно (`x1_py`, `x2_py`, `z_py`).
4. Совпадают ли результаты и что это означает для корректности решения.
